In [1]:
from datetime import date
import uuid
import math

def new_id():
    return str(uuid.uuid4())

class Mortgage:
    def __init__(
        self,
        principal=0.0,
        rate=0.0,
        start_date=None,
        term_months=None,
        monthly_payment=None,
        paid_to_date=0.0,
        due_date=None,
        status="active",
        id=None,
    ):
        self.id = id or new_id()
        self.principal = float(principal)
        self.rate = float(rate)
        self.start_date = start_date
        self.term_months = term_months
        self.monthly_payment = monthly_payment
        self.paid_to_date = float(paid_to_date)
        self.due_date = due_date
        self.status = status
        self.house = None
        self.household = None

    def calculate_monthly_payment(self):
        if not self.term_months or self.term_months <= 0:
            return None
        r = self.rate / 12.0
        n = self.term_months
        if r == 0:
            self.monthly_payment = round(self.principal / n, 2)
        else:
            self.monthly_payment = round(
                (self.principal * r) / (1 - (1 + r) ** -n), 2
            )
        return self.monthly_payment

    def remaining_balance(self):
        if self.monthly_payment is None or self.monthly_payment == 0:
            return None
        payments_made = int(self.paid_to_date / self.monthly_payment)
        outstanding = max(0.0, self.principal - payments_made * self.monthly_payment)
        return round(outstanding, 2)

    def attach_to_house(self, house):
        if self.house is not house:
            self.house = house
            house.add_mortgage(self)

    def attach_to_household(self, household):
        if self.household is not household:
            self.household = household
            household.add_mortgage(self)

    def to_dict(self):
        return {
            "id": self.id,
            "principal": self.principal,
            "rate": self.rate,
            "start_date": self.start_date.isoformat() if self.start_date else None,
            "term_months": self.term_months,
            "monthly_payment": self.monthly_payment,
            "paid_to_date": self.paid_to_date,
            "due_date": self.due_date.isoformat() if self.due_date else None,
            "status": self.status,
            "house_id": self.house.id if self.house else None,
            "household_id": self.household.id if self.household else None,
        }

class House:
    def __init__(
        self,
        address=None,
        owner=None,
        occupants=None,
        mortgages=None,
        price=None,
        rent=None,
        status="available",
        id=None,
    ):
        self.id = id or new_id()
        self.address = address
        self.owner = owner
        self.occupants = occupants or []
        self.mortgages = mortgages or []
        self.price = price
        self.rent = rent
        self.status = status
        if owner:
            owner.primary_house = self

    def add_mortgage(self, mortgage):
        if mortgage not in self.mortgages:
            self.mortgages.append(mortgage)
            mortgage.house = self

    def occupy(self, household):
        if household not in self.occupants:
            self.occupants.append(household)
            if household.primary_house is not self:
                household.primary_house = self
            self.status = "occupied"

    def remove_occupant(self, household):
        if household in self.occupants:
            self.occupants.remove(household)
            if household.primary_house is self:
                household.primary_house = None
        if not self.occupants:
            self.status = "available"

    def set_owner(self, household):
        self.owner = household
        household.primary_house = self

    def to_dict(self):
        return {
            "id": self.id,
            "address": self.address,   # temp untill attached to MEAS Cell
            "owner_id": self.owner.id if self.owner else None,
            "occupant_ids": [h.id for h in self.occupants],
            "mortgage_ids": [m.id for m in self.mortgages],
            "price": self.price,
            "rent": self.rent,
            "status": self.status,
        }

class Household:
    def __init__(
        self,
        name=None,
        members=None,
        primary_house=None,
        mortgages=None,
        id=None,
    ):
        self.id = id or new_id()
        self.name = name
        self.members = members or []
        self.primary_house = primary_house
        self.mortgages = mortgages or []
        if primary_house:
            if self not in primary_house.occupants:
                primary_house.occupants.append(self)
            primary_house.owner = self

    def add_member(self, member_name):
        if member_name not in self.members:
            self.members.append(member_name)

    def assign_house(self, house):
        self.primary_house = house
        if self not in house.occupants:
            house.occupants.append(self)
        house.owner = self
        house.status = "occupied"

    def add_mortgage(self, mortgage):
        if mortgage not in self.mortgages:
            self.mortgages.append(mortgage)
            mortgage.household = self

    def remove_mortgage(self, mortgage):
        if mortgage in self.mortgages:
            self.mortgages.remove(mortgage)
            if mortgage.household is self:
                mortgage.household = None

    def to_dict(self):
        return {
            "id": self.id,
            "name": self.name,
            "members": list(self.members),
            "primary_house_id": self.primary_house.id if self.primary_house else None,
            "mortgage_ids": [m.id for m in self.mortgages],
        }

class Realtor:
    def __init__(
        self,
        name=None,
        listings=None,
        contact=None,
        id=None,
    ):
        self.id = id or new_id()
        self.name = name
        self.listings = listings or []
        self.contact = contact

    def add_listing(self, house):
        if house not in self.listings:
            self.listings.append(house)
            house.status = "for_sale"

    def remove_listing(self, house):
        if house in self.listings:
            self.listings.remove(house)

    def to_dict(self):
        return {
            "id": self.id,
            "name": self.name,
            "listing_ids": [h.id for h in self.listings],
            "contact": self.contact,
        }

class Record:
    def __init__(
        self,
        record_type="generic",
        house=None,
        household=None,
        realtor=None,
        price=None,
        rent=None,
        mortgage=None,
        notes=None,
        created_at=None,
        id=None,
    ):
        self.id = id or new_id()
        self.created_at = created_at or date.today()
        self.record_type = record_type
        self.house = house
        self.household = household
        self.realtor = realtor
        self.price = price
        self.rent = rent
        self.mortgage = mortgage
        self.notes = notes

    def to_dict(self):
        return {
            "id": self.id,
            "created_at": self.created_at.isoformat(),
            "record_type": self.record_type,
            "house_id": self.house.id if self.house else None,
            "household_id": self.household.id if self.household else None,
            "realtor_id": self.realtor.id if self.realtor else None,
            "price": self.price,
            "rent": self.rent,
            "mortgage_id": self.mortgage.id if self.mortgage else None,
            "notes": self.notes,
        }
